# 01 — Bear coexistence groups: what the dataset is (exploration, read-only)

First notebook of the **communities** analysis. The single input is
`input_data/bear_coexistence/CoexistenceGroup_CDCounty_SpatJoin/` — one vector layer shipped twice
(`.shp` + `.gpkg`) plus `README_MetaData.docx`. Kernel `y2y-geo`. **Zero solves, nothing aligned:**
this notebook characterises the data before any decision about how (or whether) it enters the
framework. The only things it writes are two audit CSVs (`audit/`) and figures (`figures/`).

**What the metadata says** (paraphrased from `README_MetaData.docx`, printed in full in §1):

- Counts of **active local and regional bear coexistence groups**, aggregated to the **2021 Canadian
  Census Divisions (CD) and US counties** within the Y2Y boundary; desktop review + partial expert
  knowledge, **current to July 2026**. Projection WGS 84 (EPSG:4326).
- Three group types were counted — (i) agencies + ENGOs working on coexistence, (ii) Bear Smart
  communities (certified or in progress) + local community working groups, (iii) rangeland
  collaboratives + watershed groups — **but the file ships only the total**, `n_groups`.
- Excluded: groups working at federal / state / provincial level, and groups whose status is
  inactive or unknown. The authors call it "a draft and underrepresentation of all cumulative
  efforts".
- Method: each group from the Communities & Conservation tracking database was mapped as a 1 to the
  *closest* CD / county and the 1s summed per unit; **units with no groups are NA** (0 in the
  ArcGIS copies). Groups with unclear boundaries were placed by best knowledge of the town / city
  / hamlet where activity occurs.
- Fields: `country` (Canada / USA), `MappingUnit` (`<CD or county name>_<province or state>`),
  `n_groups`.

**Questions this notebook answers**

1. §1 What is in the folder, and are the `.shp` and `.gpkg` the same layer?
2. §2 What do the attributes hold — NA semantics, totals by country, which units have groups?
3. §3 Geometry — valid? already clipped to Y2Y? overlaps, gaps, and how unequal are the units?
4. §4 Are the labels right? The province/state suffix is checked against where each polygon
   actually sits (Natural Earth admin-1, the project basemap layer).
5. §5–§6 What the count looks like as a distribution and on the map, and how unit size confounds it.
6. §7 Findings + the open questions for Ethan / the Communities & Conservation team.


In [ ]:
# ---- Setup: repo root, shared config, paths --------------------------------------------------
import sys, os, re, pathlib, importlib, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Patch

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config
importlib.reload(config)

HERE = ROOT / "analyses" / "communities"
OUT_ROOT = pathlib.Path(os.environ.get("COMMUNITIES_OUT_ROOT", HERE))   # env override = headless smoke-test hook only
AUDIT, FIGS = OUT_ROOT / "audit", OUT_ROOT / "figures"
for _d in (AUDIT, FIGS):
    _d.mkdir(parents=True, exist_ok=True)

SRC_DIR = config.INPUT_DIR / "bear_coexistence" / "CoexistenceGroup_CDCounty_SpatJoin"
GPKG = SRC_DIR / "CoexistenceGroup_CDCounty_Join.gpkg"
SHP = SRC_DIR / "CoexistenceGroup_CDCounty_Join.shp"
META = SRC_DIR / "README_MetaData.docx"
ADMIN = config.INPUT_DIR / "basemap" / "ne_10m_admin_1_states_provinces.shp"          # the project basemap layer
ADMIN_LINES = config.INPUT_DIR / "basemap" / "ne_10m_admin_1_states_provinces_lines.shp"
CRS = config.TARGET_CRS                                                                   # ESRI:102008, the project grid CRS

pd.set_option("display.width", 200); pd.set_option("display.max_columns", None); pd.set_option("display.max_rows", 120)
print(f"source : {SRC_DIR.relative_to(ROOT)}")
print(f"outputs: {AUDIT.relative_to(OUT_ROOT.parent) if OUT_ROOT == HERE else AUDIT}  |  {FIGS.relative_to(OUT_ROOT.parent) if OUT_ROOT == HERE else FIGS}")
print(f"work CRS: {CRS}")


## 1 · The folder, the metadata, and the two copies of the layer

The metadata is a `.docx`; its paragraph text is extracted here so the record is in the notebook, not
only in Word. Then both copies are read and compared field-by-field and geometry-by-geometry: if they
are the same layer, everything downstream reads the `.gpkg` (no 10-character field-name truncation).

In [ ]:
# ---- Folder listing -------------------------------------------------------------------------
files = sorted(p for p in SRC_DIR.iterdir() if p.is_file())
print(f"{'file':52s} {'bytes':>12s}")
for p in files:
    note = "  <- Word lock file (junk, ignore)" if p.name.startswith("~$") else ("  <- Finder junk" if p.name == ".DS_Store" else "")
    print(f"{p.name:52s} {p.stat().st_size:12,d}{note}")

# ---- Metadata text (paragraphs of the .docx) ---------------------------------------------------
xml = zipfile.ZipFile(META).read("word/document.xml").decode("utf8")
paras = ["".join(re.findall(r"<w:t[^>]*>(.*?)</w:t>", p, flags=re.S)) for p in re.findall(r"<w:p[ >].*?</w:p>", xml, flags=re.S)]
meta_text = "\n".join(t for t in paras if t.strip())
print("\n" + "=" * 100 + "\nREADME_MetaData.docx\n" + "=" * 100 + "\n" + meta_text + "\n" + "=" * 100)


In [ ]:
# ---- Read both copies, compare ------------------------------------------------------------------
print("gpkg layers:", pyogrio.list_layers(GPKG).tolist())
g = gpd.read_file(GPKG)
s = gpd.read_file(SHP)
print(f"\ngpkg: {g.shape}  crs={g.crs.to_string()}  cols={list(g.columns)}")
print(f"shp : {s.shape}  crs={s.crs.to_string()}  cols={list(s.columns)}")

same_attr = (g["country"].equals(s["country"]) and g["MappingUnit"].equals(s["MappingUni"])
             and g["n_groups"].equals(s["n_groups"]))
exact = g.geometry.geom_equals_exact(s.geometry, tolerance=1e-9)             # same vertices in the same order
topo = g.geometry.geom_equals(s.geometry)                                     # same point set (vertex order may differ)
hausdorff_m = g.to_crs(CRS).geometry.hausdorff_distance(s.to_crs(CRS).geometry)
same_geom = bool(topo.all())
print(f"\nattributes identical (country, MappingUnit/MappingUni, n_groups): {same_attr}")
print(f"geometries: vertex-exact {exact.sum()}/{len(g)}   topologically equal {topo.sum()}/{len(g)}   "
      f"max Hausdorff distance {hausdorff_m.max():.3f} m")
print("-> the same layer (only the ring start / vertex order differs); downstream reads the .gpkg"
      if (same_attr and same_geom) else "-> DIFFER: inspect before choosing a copy")
print("\nschema (gpkg):"); print(g.dtypes.to_string())


## 2 · Attributes

`n_groups` is NA where no group was recorded (the metadata's rule), so the honest count of "units with
at least one group" is `notna()`, and totals are sums over the non-NA rows. `MappingUnit` should be a
unique key — it is checked.

In [ ]:
n_units = len(g)
has = g["n_groups"].notna()
print(f"units: {n_units}   with >=1 group: {has.sum()}   NA (no group recorded): {(~has).sum()}   "
      f"total groups (as shipped): {g['n_groups'].sum():.0f}")
print("\nn_groups value counts (NA = no group recorded):")
print(g["n_groups"].value_counts(dropna=False).sort_index(na_position="first").rename_axis("n_groups").to_frame("units").T.to_string())

by_country = g.assign(has=has).groupby("country").agg(units=("MappingUnit", "size"), units_with_groups=("has", "sum"),
                                                     groups=("n_groups", "sum"))
by_country["groups"] = by_country["groups"].astype(int)
print("\nby country:"); print(by_country.to_string())

dup = g["MappingUnit"].duplicated(keep=False)
print(f"\nMappingUnit unique: {g['MappingUnit'].nunique()} of {n_units}  ->  {dup.sum()} rows share a name with another row "
      f"({g.loc[dup, 'MappingUnit'].nunique()} names). Checked in §4.")
if dup.any():
    print(g.loc[dup].drop(columns="geometry").sort_values("MappingUnit").to_string())

print("\nunits with >=1 group (as shipped, descending):")
print(g.loc[has].drop(columns="geometry").sort_values(["n_groups", "country", "MappingUnit"], ascending=[False, True, True])
       .reset_index(drop=True).to_string())


## 3 · Geometry

Reprojected in memory to the project CRS (ESRI:102008, equal-area, so km² are honest). Checks: validity,
parts per feature, unit sizes, overlaps between units (sum of areas vs area of the union), and the
relationship to the **unbuffered** 2013 Y2Y boundary (`config.CORRIDOR_REF`): are the units full
CDs / counties, or already clipped to Y2Y? If clipped, how much of Y2Y is left uncovered?

In [ ]:
ga = g.to_crs(CRS).copy()
ga["area_km2"] = ga.area / 1e6
ga["n_parts"] = ga.geometry.apply(lambda x: len(getattr(x, "geoms", [x])))
y2y = gpd.read_file(config.CORRIDOR_REF).to_crs(CRS).union_all()
units_union = ga.union_all()

print(f"geometry types : {ga.geom_type.value_counts().to_dict()}")
print(f"valid          : {ga.is_valid.sum()} / {len(ga)}    empty: {ga.is_empty.sum()}")
print(f"parts/feature  : {ga['n_parts'].value_counts().sort_index().to_dict()}")
print(f"\nunit area km2  : min {ga.area_km2.min():,.2f}   median {ga.area_km2.median():,.0f}   mean {ga.area_km2.mean():,.0f}   "
      f"max {ga.area_km2.max():,.0f}   (ratio max/min {ga.area_km2.max()/ga.area_km2.min():,.0f}x)")
print(f"slivers < 1 km2: {(ga.area_km2 < 1).sum()}   units < 100 km2: {(ga.area_km2 < 100).sum()}")
print("\nlargest 5:");  print(ga.nlargest(5, "area_km2")[["country", "MappingUnit", "n_groups", "area_km2"]].to_string())
print("\nsmallest 5:"); print(ga.nsmallest(5, "area_km2")[["country", "MappingUnit", "n_groups", "area_km2"]].to_string())

overlap_km2 = ga.area_km2.sum() - units_union.area / 1e6
print(f"\nsum of unit areas {ga.area_km2.sum():,.0f} km2  vs  union {units_union.area/1e6:,.0f} km2  ->  overlap {overlap_km2:,.1f} km2 "
      f"({overlap_km2 / ga.area_km2.sum():.2%})")

# relationship to the Y2Y boundary
ga["in_y2y_km2"] = ga.geometry.intersection(y2y).area / 1e6
ga["frac_in_y2y"] = ga["in_y2y_km2"] / ga["area_km2"]
covered = units_union.intersection(y2y).area / 1e6
gap = y2y.difference(units_union)
gap_parts = list(getattr(gap, "geoms", [gap]))
gap_parts = sorted((p for p in gap_parts if not p.is_empty), key=lambda p: -p.area)
print(f"\nY2Y (unbuffered 2013) area {y2y.area/1e6:,.0f} km2;  covered by the units {covered:,.0f} km2 ({covered/(y2y.area/1e6):.4%})")
print(f"per-unit share inside Y2Y: min {ga.frac_in_y2y.min():.4f}  median {ga.frac_in_y2y.median():.4f}  "
      f"units with < 99% inside: {(ga.frac_in_y2y < 0.99).sum()}")
print(f"-> {'the units are ALREADY CLIPPED to the Y2Y boundary (nothing extends outside it)' if ga.frac_in_y2y.min() > 0.98 else 'the units extend beyond Y2Y (full CD/county polygons)'}")
if gap_parts:
    c = gpd.GeoSeries([gap_parts[0].centroid], crs=CRS).to_crs(4326).iloc[0]
    print(f"uncovered Y2Y: {gap.area/1e6:,.1f} km2 in {len(gap_parts)} pieces; largest {gap_parts[0].area/1e6:,.1f} km2 near "
          f"({c.y:.2f} N, {abs(c.x):.2f} W); pieces >= 1 km2: {sum(p.area/1e6 >= 1 for p in gap_parts)}")
outside = ga.area_km2.sum() - ga.in_y2y_km2.sum()
print(f"unit area outside Y2Y: {outside:,.1f} km2 total ({outside / ga.area_km2.sum():.4%})")


## 4 · Label QA — does the province/state suffix match where the polygon sits?

`MappingUnit` is `<name>_<province or state>`. The suffix is compared with the Natural Earth admin-1
polygon containing each unit's representative point (the same layer the director basemaps use). A
mismatch means the *label* is wrong, not necessarily the polygon. Two derived flags:

- `state_label_ok` — suffix equals the containing admin-1 name;
- `name_collision` — the same `MappingUnit` string is attached to two different polygons. Where a
  mislabelled unit collides with a genuine same-name unit in the neighbouring state, its `n_groups`
  can only be trusted if it was counted independently — the metadata's "mapped to the closest
  CD/county" step gives no way to tell, so both totals (as shipped, and with the collided copies
  set aside) are reported.

In [ ]:
adm = gpd.read_file(ADMIN, columns=["name", "admin", "iso_a2"])
adm = adm[adm["iso_a2"].isin(["CA", "US"])].to_crs(CRS)[["name", "admin", "geometry"]]
pts = ga[["MappingUnit", "geometry"]].copy(); pts["geometry"] = ga.representative_point()
hit = gpd.sjoin(pts, adm, how="left", predicate="within").drop(columns=["index_right", "geometry"])
assert not hit.index.duplicated().any(), "a representative point fell in two admin-1 polygons"

ga["name"] = ga["MappingUnit"].str.rsplit("_", n=1).str[0]
ga["label_state"] = ga["MappingUnit"].str.rsplit("_", n=1).str[1]
ga["ne_state"] = hit["name"]
ga["ne_country"] = hit["admin"].map({"Canada": "Canada", "United States of America": "USA"})
ga["state_label_ok"] = ga["label_state"] == ga["ne_state"]
ga["country_ok"] = ga["country"] == ga["ne_country"]
ga["name_collision"] = ga["MappingUnit"].duplicated(keep=False)

print(f"admin-1 found for {ga['ne_state'].notna().sum()} / {len(ga)} units")
print(f"country field matches admin-1     : {ga.country_ok.sum()} / {len(ga)}")
print(f"province/state suffix matches     : {ga.state_label_ok.sum()} / {len(ga)}   ->  {(~ga.state_label_ok).sum()} MISLABELLED")
print("\nunits present, by admin-1 (where the polygons actually are):")
print(ga.groupby(["ne_country", "ne_state"]).agg(units=("MappingUnit", "size"), with_groups=("n_groups", lambda x: x.notna().sum()),
                                                  groups=("n_groups", lambda x: int(x.sum()))).to_string())

bad = ga.loc[~ga.state_label_ok, ["country", "MappingUnit", "label_state", "ne_state", "n_groups", "area_km2", "name_collision"]] \
        .sort_values(["ne_state", "MappingUnit"])
print(f"\nmislabelled units ({len(bad)}):")
print(bad.to_string())

# the collisions: same label, different polygon
coll = ga.loc[ga.name_collision, ["MappingUnit", "label_state", "ne_state", "n_groups", "area_km2", "state_label_ok"]].sort_values(["MappingUnit", "state_label_ok"], ascending=[True, False])
print(f"\nname collisions ({coll['MappingUnit'].nunique()} names, {len(coll)} rows) - the mislabelled copy carries the SAME n_groups as the genuine one:")
print(coll.to_string())
suspect = ga.name_collision & ~ga.state_label_ok
print(f"\ntotal groups as shipped: {ga.n_groups.sum():.0f}   |   with the {suspect.sum()} collided mislabelled copies set aside: "
      f"{ga.loc[~suspect, 'n_groups'].sum():.0f}   (difference {ga.loc[suspect, 'n_groups'].sum():.0f}, on {suspect.sum()} units)")

# ---- audit tables -----------------------------------------------------------------------------
cols = ["country", "MappingUnit", "name", "label_state", "ne_state", "state_label_ok", "name_collision", "n_groups",
        "area_km2", "in_y2y_km2", "frac_in_y2y", "n_parts"]
units_tbl = ga[cols].copy().round({"area_km2": 2, "in_y2y_km2": 2, "frac_in_y2y": 5})
units_tbl.to_csv(AUDIT / "bear_coexistence_units.csv", index_label="fid")
bad.round({"area_km2": 2}).to_csv(AUDIT / "bear_coexistence_label_qa.csv", index_label="fid")
print(f"\nwrote {AUDIT / 'bear_coexistence_units.csv'}  ({len(units_tbl)} rows)")
print(f"wrote {AUDIT / 'bear_coexistence_label_qa.csv'}  ({len(bad)} rows)")


## 5 · Distributions

Three views of the same attribute: how many units fall in each count class; which units hold the groups
(labelled by the admin-1 the polygon sits in, so mislabels read correctly); and the unit-size spread —
the reason a raw count is not comparable across units.

In [ ]:
BLUE, GREY, INK, MUTED = "#2F6DB5", "#C9CCD1", "#1F2933", "#6B7280"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False})

fig, axes = plt.subplots(1, 3, figsize=(15, 7.5), gridspec_kw={"width_ratios": [1.0, 1.7, 1.2]})

# (a) units per count class
ax = axes[0]
counts = ga["n_groups"].fillna(0).astype(int).value_counts().sort_index()
ax.bar(counts.index.astype(str), counts.values, color=[GREY if k == 0 else BLUE for k in counts.index], width=0.7)
for x, v in zip(counts.index.astype(str), counts.values):
    ax.text(x, v + 0.8, str(v), ha="center", va="bottom", color=INK)
ax.set_xlabel("groups recorded in the unit (0 = NA)"); ax.set_ylabel("units"); ax.set_title("(a) units per count class", loc="left")
ax.grid(axis="y", color="#E5E7EB", lw=0.6); ax.set_axisbelow(True)

# (b) the units with groups
ax = axes[1]
top = ga.loc[ga.n_groups.notna()].sort_values(["n_groups", "area_km2"], ascending=[True, True])
lab = top["name"] + "  (" + top["ne_state"].fillna("?") + (np.where(top["state_label_ok"], "", " — label says " + top["label_state"])) + ")"
ax.barh(lab, top["n_groups"], color=BLUE, height=0.7)
ax.set_xlabel("groups"); ax.set_title("(b) units with >=1 group — shown by the admin-1 the polygon sits in", loc="left")
ax.set_xticks([0, 1, 2, 3, 4]); ax.grid(axis="x", color="#E5E7EB", lw=0.6); ax.set_axisbelow(True); ax.tick_params(axis="y", labelsize=8)

# (c) unit size spread
ax = axes[2]
bins = np.logspace(-2, 6, 33)
ax.hist(ga["area_km2"], bins=bins, color=GREY, label="all units")
ax.hist(ga.loc[ga.n_groups.notna(), "area_km2"], bins=bins, color=BLUE, label="units with >=1 group")
ax.set_xscale("log"); ax.set_xlabel("unit area inside Y2Y (km², log)"); ax.set_ylabel("units")
ax.set_title("(c) unit sizes span 7 orders of magnitude", loc="left"); ax.legend(frameon=False)
ax.grid(axis="y", color="#E5E7EB", lw=0.6); ax.set_axisbelow(True)

fig.suptitle("Bear coexistence groups by CD / county — attribute distributions", x=0.01, ha="left", fontsize=12, color=INK)
fig.tight_layout()
fig.savefig(FIGS / "bear_coexistence_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## 6 · Map

Left: the shipped count (one hue, light → dark; NA units grey). Right: the same numbers as a density,
groups per 10,000 km² of unit area, which is what a per-cell layer would inherit if the count were
simply painted onto the grid. Units with three or more groups are named on the left panel. Context:
the Y2Y outline and the Natural Earth admin-1 boundaries.

In [ ]:
lines = gpd.read_file(ADMIN_LINES).to_crs(CRS)
xmin, ymin, xmax, ymax = ga.total_bounds
pad = 60e3
lines = gpd.clip(lines, (xmin - pad, ymin - pad, xmax + pad, ymax + pad))
y2y_gs = gpd.GeoSeries([y2y], crs=CRS)

steps = ["#C6DBEF", "#6BAED6", "#2171B5", "#08306B"]          # one hue, 4 steps, light -> dark (counts 1..4)
cmap4 = mcolors.ListedColormap(steps); norm4 = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], 4)
na = ga.n_groups.isna()
ga["density_per_1e4km2"] = ga["n_groups"] / ga["area_km2"] * 1e4

fig, axes = plt.subplots(1, 2, figsize=(12, 9))
for ax in axes:
    ga.loc[na].plot(ax=ax, color="#EEEFF1", edgecolor="white", linewidth=0.4)
    lines.plot(ax=ax, color=MUTED, linewidth=0.5, linestyle=(0, (4, 2)), zorder=3)
    y2y_gs.boundary.plot(ax=ax, color=INK, linewidth=0.9, zorder=4)
    ax.set_axis_off()

ax = axes[0]
ga.loc[~na].plot(ax=ax, column="n_groups", cmap=cmap4, norm=norm4, edgecolor="white", linewidth=0.4, zorder=2)
LABEL_OFFSET = {"Central Kootenay": (-34, -4), "East Kootenay": (34, 12)}      # (points) to keep neighbours apart
for _, r in ga.loc[ga.n_groups >= 3].iterrows():
    p = r.geometry.representative_point()
    dx, dy = LABEL_OFFSET.get(r["name"], (28, 8))
    ax.annotate(f"{r['name']} ({int(r.n_groups)})", (p.x, p.y), xytext=(dx, dy), textcoords="offset points", fontsize=8, color=INK,
                ha="right" if dx < 0 else "left", arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6), zorder=6)
handles = [Patch(facecolor=c, edgecolor="white", label=f"{k} group{'s' if k > 1 else ''}") for k, c in zip(range(1, 5), steps)]
handles.append(Patch(facecolor="#EEEFF1", edgecolor="#C9CCD1", label="none recorded (NA)"))
ax.legend(handles=handles, loc="lower left", frameon=False, title="groups per unit", fontsize=8, title_fontsize=9)
ax.set_title("(a) n_groups as shipped", loc="left")

ax = axes[1]
vmax = float(ga["density_per_1e4km2"].max())
ga.loc[~na].plot(ax=ax, column="density_per_1e4km2", cmap="Blues", vmin=0, vmax=vmax, edgecolor="white", linewidth=0.4, zorder=2,
                 legend=True, legend_kwds={"shrink": 0.45, "label": "groups per 10,000 km² of unit", "pad": 0.01})
ax.set_title("(b) the same counts as a density — unit size dominates", loc="left")

fig.suptitle("Bear coexistence groups by 2021 CD / US county, clipped to Y2Y (ESRI:102008)", x=0.01, ha="left", fontsize=12, color=INK)
fig.tight_layout()
fig.savefig(FIGS / "bear_coexistence_map.png", dpi=150, bbox_inches="tight")
plt.show()

print(ga.loc[~na].nlargest(8, "density_per_1e4km2")[["name", "ne_state", "n_groups", "area_km2", "density_per_1e4km2"]].round(2).to_string())


## 7 · Findings and open questions

**What the layer is.** A **Y2Y-clipped tessellation**, not a set of whole CDs / counties: 108 polygons
that cover 99.99% of the unbuffered 2013 Y2Y boundary, with 2.2 km² outside it, 3 km² of mutual overlap, all
valid. The uncovered 144 km² is 119 slivers along the boundary plus one 133 km² piece at the Alaska corner
(~65.3° N, 141.0° W) — there is no Alaska unit in the file. Unit sizes run from 0.02 km² (Canyon County ID,
a clip sliver) to 296,959 km² (the Yukon CD), median 3,874 km².

**The `.shp` and `.gpkg` are the same layer** (attributes identical; geometries topologically identical,
Hausdorff distance 0 m — only the vertex order differs, so an exact comparison fails). The shapefile
truncates `MappingUnit` to `MappingUni`. Everything downstream should read the `.gpkg`.

**Attributes.** 30 of 108 units hold at least one group; 53 groups as shipped (Canada 18 in 9 units, USA 35
in 21). Counts run 1–4; Central Kootenay (BC) and Missoula County (MT) top the list at 4. 78 units are NA **and there
are no zeros in the file**; the metadata says NA = "no group recorded" (shown as 0 in the team's ArcGIS copies).
**Decision (Ethan, 2026-09-16): NA stays NA, not recoded to 0** — "no record" is never painted as a measured zero,
and the 17 mislabelled NA units keep the possibility of a lost count visible.

**Label QA — 21 of 108 units carry the wrong province/state suffix** (`audit/bear_coexistence_label_qa.csv`).
In every case the CD / county *name* is a real unit of the admin-1 that contains the polygon
(Gallatin / Ravalli / Sanders → Montana; Baker / Malheur / Wallowa → Oregon; Asotin / Columbia / Garfield /
Pend Oreille / Spokane / Walla Walla → Washington; Regions 1, 2, 4 → NWT; Yukon → Yukon; Peace River → BC),
so **the polygons are right and the suffix is wrong** — it looks taken from a neighbouring jurisdiction rather
than the containing one. Four mislabels collide with a genuine same-name county next door (Madison ID vs MT,
Teton ID vs WY, Lincoln WY vs MT, Park WY vs MT) and **each carries exactly its namesake's `n_groups`** — the
signature of a name-keyed attribution in the "mapped to the closest CD / county" step. If those four are
copies, the true total is **47, not 53**, and the top of the density panel (Madison County ID, 28.5 groups per
10,000 km²) is an artefact. Flagged, not fixed: the fix belongs upstream, in the tracking database export.

**Support and representation.** A count per unit is not comparable across units whose areas span seven
orders of magnitude (§5c, §6b): one group in a 700 km² county and one in the 297,000 km² Yukon CD paint the
same colour on panel (a) and a 400× different one on panel (b). The metadata says groups were placed by the
town / city / hamlet where activity occurs — the CD / county polygon is an administrative *support*, not where
the effort is. Before this becomes anything per-cell the representation has to be chosen: presence / absence,
count, density, or the group locations themselves (points from the tracking database).

**Other notes.** Current to July 2026, and "a draft and underrepresentation" by the authors' own statement
(federal / provincial / state-level groups and inactive / unknown-status groups excluded). Only the total
ships — the three group types described in the metadata are not in the file. The metadata's file-name field
reads "…_JoinFire" (stale). `~$ADME_MetaData.docx` is a Word lock file; `input_data/` is gitignored, so none
of this is tracked.

**Open questions (decisions, not for this notebook)**

1. Send the label QA back to the Communities & Conservation team: confirm whether Madison ID, Teton ID,
   Lincoln WY and Park WY really have groups, and re-export with the suffix from the containing polygon.
2. Per-type counts (agencies / ENGOs · Bear Smart + local groups · rangeland + watershed collaboratives) and,
   ideally, the group point locations — if type or place matters to the analysis.
3. The representation on the grid (presence, count, density, or points). NA stays NA (decided).
4. The role in the framework: this is a *capacity* layer (where people are already working on coexistence),
   not a conservation value like the stack's features — whether it enters as a feature, a cost / opportunity
   modifier, or a post-hoc overlay on the tiers is a spec decision to take with the spec, not here.
